# Setting up / importing

In [ ]:
import sys

PROJECT_DIR = "/home/565/pv3484/aus_substation_electricity"
sys.path.append(PROJECT_DIR)
%cd {PROJECT_DIR}

In [ ]:
%run /home/565/pv3484/aus_substation_electricity/import_substation.py

In [ ]:
import time

import pandas as pd
from dateutil.easter import easter
from geopy.geocoders import Nominatim

RANK_CSV = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"


def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y (Monarch's Birthday)."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]
    return mondays[1]


# NSW public holidays, including Easter (date-calculated each year)
HOLIDAYS_VIC = {
    "New Year's Day":  lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day":   lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday":     lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday":   lambda y: pd.Timestamp(easter(y)),
    "Easter Monday":   lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day":       lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day":   lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day":      lambda y: pd.Timestamp(f"{y}-12-26"),
}

holiday_order = list(HOLIDAYS_VIC.keys())

# Maps individual holiday names to their group label
HOLIDAY_GROUPS = {
    "Good Friday":     "Easter Long Weekend",
    "Easter Saturday": "Easter Long Weekend",
    "Easter Sunday":   "Easter Long Weekend",
    "Easter Monday":   "Easter Long Weekend",
    "Christmas Day":   "Christmas and Boxing Day",
    "Boxing Day":      "Christmas and Boxing Day",
}


In [ ]:
info.loc[info["Name"] == "Chatswood", "Residential"]

In [ ]:
rank = pd.read_csv(RANK_CSV)

## Geocode substations

Look up lat/lon for each substation by suburb name. Results are written back into `info`.
Manually patches **Dee Why West**, which has no polygon in the geocoder.

In [ ]:
geolocator = Nominatim(user_agent="sydney_demand_mapper")


def get_coords(place):
    """Return (lat, lon) for a NSW suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None


latitudes, longitudes = [], []
for suburb in info["Name"]:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # respect Nominatim rate limit

info["latitude"] = latitudes
info["longitude"] = longitudes

In [ ]:
# Check which stations failed to geocode
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()

In [ ]:
# Dee Why West has no suburb polygon — patch with manually sourced coordinates
info.loc[info["Name"] == "Dee Why West", "latitude"] = -33.73441
info.loc[info["Name"] == "Dee Why West", "longitude"] = 151.28278

## Map mean relative rank differences

Three-panel map showing, for each high-residential substation:
- **Panel 1** – Public holiday mean relative rank (binned, plasma palette)
- **Panel 2** – PH minus weekend (separate scale, RdBu_r)
- **Panel 3** – PH minus weekday (separate scale, PuOr_r)

In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import contextily as ctx
import numpy as np
import pathlib
import pandas as pd
from matplotlib.colors import ListedColormap

BLOCK_LABELS = {
    "00_04": "00:00–04:00",
    "04_10": "04:00–10:00",
    "10_15": "10:00–15:00",
    "15_20": "15:00–20:00",
    "20_24": "20:00–00:00",
}

ALL_BLOCKS = list(BLOCK_LABELS.keys())

HOLIDAY_BATCH = [
    ("New Year's Day",           ["New Year's Day"]),
    ("Australia Day",            ["Australia Day"]),
    ("Easter Long Weekend",      ["Good Friday", "Easter Saturday", "Easter Sunday", "Easter Monday"]),
    ("ANZAC Day",                ["ANZAC Day"]),
    ("Queen's Birthday",         ["Monarch's Birthday"]),
    ("Christmas and Boxing Day", ["Christmas Day", "Boxing Day"]),
]

FIG_ROOT = pathlib.Path(
    "/home/565/pv3484/aus_substation_electricity/data/figures"
    "/mrr_pb_minus_we_wd/separate_scale_new_scale"
)


def map_ph_we_wd_block_combined(
    df,
    info,
    holidays,
    holiday_group_name,
    block,
    we_diff_min=None,
    we_diff_max=None,
    wd_diff_min=None,
    wd_diff_max=None,
    df_station_col="Name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude",
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean_relative_rank",
    crs_epsg=3857,
    wd_diff_cmap="PuOr_r",
):
    """Three-panel map of public holiday relative demand vs. weekends and weekdays.

    Parameters
    ----------
    df : DataFrame
        Relative rank data (one row per station-date).
    info : DataFrame
        Substation metadata including lat/lon and land-use fractions.
    holidays : list[str]
        Holiday names to treat as the public holiday group.
    holiday_group_name : str
        Display name for the holiday group (used in the title).
    block : str
        Time block key, e.g. ``"10_15"``.
    we_diff_min, we_diff_max : float, optional
        Colour scale limits for the PH−weekend panel. Auto-symmetrised if omitted.
    wd_diff_min, wd_diff_max : float, optional
        Colour scale limits for the PH−weekday panel. Auto-symmetrised if omitted.
    """
    block_label = BLOCK_LABELS.get(block, block)

    # --- Eligible stations: high-residential, with data, excluding Umina ---
    high_res = info[
        (info[residential_col] >= residential_threshold)
        & (info[info_station_col] != "Umina")
    ][info_station_col]

    block_cols = [f"{b}{block_suffix}" for b in ALL_BLOCKS]
    stations_with_data = (
        df.groupby(df_station_col)[block_cols]
        .apply(lambda x: x.notna().any().any())
    )
    stations_with_data = stations_with_data[stations_with_data].index
    eligible_stations = set(high_res).intersection(stations_with_data)

    df = df[df[df_station_col].isin(eligible_stations)].copy()
    df_hol = df[df["holiday_group"].isin(holidays)].copy()

    # --- GeoDataFrame ---
    gdf = info[info[info_station_col].isin(eligible_stations)].copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326",
    ).to_crs(crs_epsg)

    # --- Compute PH, PH−WE, PH−WD per station ---
    col = f"{block}{block_suffix}"
    results = []

    for station in eligible_stations:
        df_s = df_hol[df_hol[df_station_col] == station]

        if df_s.empty or df_s[col].isna().all():
            results.append({info_station_col: station, "ph": np.nan, "ph_we": np.nan, "ph_wd": np.nan})
            continue

        df_station_all = df[df[df_station_col] == station]

        ph_val = df_s[df_s["is_holiday"]][col].mean()
        we_val = df_station_all[~df_station_all["is_holiday"] & df_station_all["is_weekend"]][col].mean()
        wd_val = df_station_all[~df_station_all["is_holiday"] & ~df_station_all["is_weekend"]][col].mean()

        results.append({
            info_station_col: station,
            "ph":    ph_val,
            "ph_we": ph_val - we_val if pd.notna(ph_val) and pd.notna(we_val) else np.nan,
            "ph_wd": ph_val - wd_val if pd.notna(ph_val) and pd.notna(wd_val) else np.nan,
        })

    gdf = gdf.merge(pd.DataFrame(results), on=info_station_col, how="left")

    # Guard against empty data
    if gdf["ph"].isna().all():
        plt.close("all")
        raise ValueError(f"No data for holiday='{holiday_group_name}', block='{block}'")

    # --- PH bins and plasma palette ---
    bins = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    bin_labels = [f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins) - 1)]

    gdf["ph_clipped"] = gdf["ph"].clip(lower=bins[0], upper=bins[-1])
    gdf["ph_bin"] = pd.cut(gdf["ph_clipped"], bins=bins, labels=bin_labels, include_lowest=True)

    plasma = matplotlib.colormaps.get_cmap("plasma")
    ph_colors = [plasma(p) for p in np.linspace(0, 1, len(bin_labels)) ** 0.7]
    ph_cmap = ListedColormap(ph_colors)

    # --- Auto-symmetrise difference scales if not provided ---
    if we_diff_min is None:
        abs_we = max(abs(gdf["ph_we"].min()), abs(gdf["ph_we"].max()))
        we_diff_min, we_diff_max = -abs_we, abs_we
    if wd_diff_min is None:
        abs_wd = max(abs(gdf["ph_wd"].min()), abs(gdf["ph_wd"].max()))
        wd_diff_min, wd_diff_max = -abs_wd, abs_wd

    # --- Map extent with padding ---
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 4800
    extent = (minx - pad, maxx + pad, miny - pad, maxy + pad)

    # --- Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax in axes:
        ax.set_aspect("equal", adjustable="box")

    year_min = int(df_hol["year"].min()) if not df_hol.empty else "?"
    year_max = int(df_hol["year"].max()) if not df_hol.empty else "?"

    fig.suptitle(
        f"Relative Demand Patterns ({holiday_group_name}) "
        f"({block_label}, {year_min}–{year_max})\n"
        f"High-residential locations only (≥ {residential_threshold})",
        fontsize=17, y=0.999,
    )

    title_y = 0.99
    axes[0].set_title("Public Holiday Relative Rank (binned)", y=title_y)
    axes[1].set_title("Public Holiday − Weekend Mean Relative Rank", y=title_y)
    axes[2].set_title("Public Holiday − Weekday Mean Relative Rank", y=title_y)

    MARKER_KWARGS = dict(markersize=70, edgecolor="black", legend=False,
                         missing_kwds={"color": "#e6e6e6", "label": "No data"})

    # Panel 1: PH binned
    gdf.plot(ax=axes[0], column="ph_bin", cmap=ph_cmap, **MARKER_KWARGS)

    handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=ph_colors[i], markersize=10)
        for i in range(len(bin_labels))
    ]
    handles.append(
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e6e6e6", markersize=10, label="No data")
    )
    fig.legend(
        handles, bin_labels + ["No data"],
        title="Public Holiday Relative Rank (binned)\n(values > 0.60 clipped to top bin)",
        loc="center left", bbox_to_anchor=(0.005, 0.50), frameon=False, fontsize=8,
    )

    # Panel 2: PH − Weekend
    gdf.plot(ax=axes[1], column="ph_we", cmap="RdBu_r", vmin=we_diff_min, vmax=we_diff_max, **MARKER_KWARGS)
    sm_we = plt.cm.ScalarMappable(cmap="RdBu_r", norm=plt.Normalize(vmin=we_diff_min, vmax=we_diff_max))
    sm_we._A = []
    fig.colorbar(sm_we, ax=axes[1], orientation="horizontal", fraction=0.05, pad=0.02).set_label(
        f"PH − Weekend ({we_diff_min:.2f} to {we_diff_max:.2f})"
    )

    # Panel 3: PH − Weekday
    gdf.plot(ax=axes[2], column="ph_wd", cmap=wd_diff_cmap, vmin=wd_diff_min, vmax=wd_diff_max, **MARKER_KWARGS)
    sm_wd = plt.cm.ScalarMappable(cmap=wd_diff_cmap, norm=plt.Normalize(vmin=wd_diff_min, vmax=wd_diff_max))
    sm_wd._A = []
    fig.colorbar(sm_wd, ax=axes[2], orientation="horizontal", fraction=0.05, pad=0.02).set_label(
        f"PH − Weekday ({wd_diff_min:.2f} to {wd_diff_max:.2f})"
    )

    # Basemap and station labels
    for ax in axes:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
        for _, row in gdf.iterrows():
            ax.text(row.geometry.x + 800, row.geometry.y, row[info_station_col],
                    fontsize=8, ha="left", va="center")
        ax.set_axis_off()

    plt.subplots_adjust(top=0.86, bottom=0.16, wspace=0.08, hspace=0.01)
    return fig


def compute_global_diff_scales(df, holiday_batch, block_suffix="_mean_relative_rank",
                                df_station_col="Name"):
    """Compute symmetric colour scale limits for PH−WE and PH−WD across all holidays and blocks."""
    all_we, all_wd = [], []

    for _, holidays in holiday_batch:
        df_hol = df[df["holiday_group"].isin(holidays)]

        for block in ALL_BLOCKS:
            col = f"{block}{block_suffix}"

            for station in df[df_station_col].unique():
                df_s = df_hol[df_hol[df_station_col] == station]
                df_all = df[df[df_station_col] == station]

                ph_val = df_s[df_s["is_holiday"]][col].mean() if not df_s.empty else np.nan
                we_val = df_all[~df_all["is_holiday"] & df_all["is_weekend"]][col].mean()
                wd_val = df_all[~df_all["is_holiday"] & ~df_all["is_weekend"]][col].mean()

                if pd.notna(ph_val) and pd.notna(we_val):
                    all_we.append(ph_val - we_val)
                if pd.notna(ph_val) and pd.notna(wd_val):
                    all_wd.append(ph_val - wd_val)

    abs_we = max(abs(min(all_we)), abs(max(all_we)))
    abs_wd = max(abs(min(all_wd)), abs(max(all_wd)))

    return (-abs_we, abs_we), (-abs_wd, abs_wd)


In [ ]:
fig = map_ph_we_wd_block_combined(
    df=rank,
    info=info,
    holidays=["ANZAC Day"],
    holiday_group_name="ANZAC Day",
    block="10_15",
)
plt.show()

## looping and saving

In [ ]:
def batch_export_mrr_maps(df, info, holiday_batch=HOLIDAY_BATCH, fig_root=FIG_ROOT):
    """Generate and save one PNG per holiday group × time block.

    Output layout::

        <fig_root>/<Holiday Group Name>/<block>.png
    """
    (we_min, we_max), (wd_min, wd_max) = compute_global_diff_scales(df, holiday_batch)
    print(f"Global scales — PH−WE: [{we_min:.3f}, {we_max:.3f}]  PH−WD: [{wd_min:.3f}, {wd_max:.3f}]")

    total = len(holiday_batch) * len(ALL_BLOCKS)
    done = 0

    for group_name, holidays in holiday_batch:
        folder_name = group_name.replace("/", "-").replace("'", "")
        out_dir = fig_root / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)

        for block in ALL_BLOCKS:
            out_path = out_dir / f"{block}.png"
            try:
                fig = map_ph_we_wd_block_combined(
                    df=df,
                    info=info,
                    holidays=holidays,
                    holiday_group_name=group_name,
                    block=block,
                    we_diff_min=we_min,
                    we_diff_max=we_max,
                    wd_diff_min=wd_min,
                    wd_diff_max=wd_max,
                )
            except ValueError as e:
                print(f"  [SKIP] {e}")
                done += 1
                continue

            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            done += 1
            print(f"[{done}/{total}] Saved: {out_path}")

    print("\nDone.")

In [ ]:
batch_export_mrr_maps(df=rank, info=info)